# Document-level unsupervised comparison

Select the project kernel and use **Run All** to launch the complete local comparison. Run All starts model fitting. Each seed has its own execution cell and the parameter registry is already defined. The final cells create the scorecard and comparison figures. [Research questions and hypotheses](15_unsupervised_research_plan_and_local_pilot.ipynb)

This experiment uses the v1 training exploration cohort from Notebook 13. The input contains 20,000 sampled parents, including two with zero retained lexical features. All six pipelines use the same remaining 19,998 training documents and a single 5,000-document validation panel. The test split is reserved. The formal successor training target is a separate release question. This experiment uses no reference chunk assets for fitting and does not change the chunk release gate.

The frozen MiniLM encoder processes each complete parent in model-budget input windows and pools every content token to one document vector. The window boundaries are internal to encoding. Training, comparison and storage of assignments each use one row per document. This run uses local model weights and no API.

## Registered comparison

| Pipeline | Feature space | Three configurations |
| --- | --- | --- |
| SVD + KMeans | Frozen word and bigram TF–IDF, reduced to 100 dimensions | 8, 12, 16 clusters |
| NMF | Frozen sparse TF–IDF | 8, 12, 16 components |
| LDA | Word and bigram count vectors on the same vocabulary | 8, 12, 16 topics |
| MiniLM + KMeans | Full-parent pooled frozen MiniLM vectors | 8, 12, 16 clusters |
| BERTopic | The same frozen MiniLM document vectors, with UMAP, HDBSCAN and topic representation | Minimum cluster size 20, 50, 100 |
| SVD + UMAP + HDBSCAN | The lexical SVD vectors | Minimum cluster size 20, 50, 100 |

Each of the three seed cells fits 18 configurations. The run manifest records all 54 expected fits. SVD is fitted only to the training cohort, and the frozen encoder vectors are shared across the embedding-based configurations.

Selection is a later research decision. Screen for viable coverage and numerical fit, then assess seed stability, coherence and representative evidence. Record uncertainty and corpus sensitivity before preferring a configuration. Reconstruction error and perplexity are model-family diagnostics. CH, DB and silhouette use a common training-fitted SVD space on validation documents. Report coverage alongside geometry metrics. [BERTopic pipeline design](https://maartengr.github.io/BERTopic/algorithm/algorithm.html)

In [ ]:
import json
import os
import platform
import tempfile
from functools import lru_cache
from importlib.metadata import version
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from scipy.sparse import csr_matrix

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/15_unsupervised_research_plan_and_local_pilot.ipynb").is_file()
)
NOTEBOOK = ROOT / "notebooks/16_document_unsupervised_comparison.ipynb"
n15 = nbformat.read(ROOT / "notebooks/15_unsupervised_research_plan_and_local_pilot.ipynb", as_version=4)
upstream15 = {"__name__": "__main__"}
input_cell = next(c.source for c in n15.cells if c.id == "15-input")
exec(compile(input_cell, "15_unsupervised_research_plan_and_local_pilot.ipynb:15-input", "exec"), upstream15)
sha256_file = upstream15["sha256_file"]
digest = upstream15["canonical_sha256"]
vocabulary = upstream15["vocabulary"]
vocab_words = vocabulary["term"].to_numpy(dtype=str)
vocab_index = {term: i for i, term in enumerate(vocab_words)}
train_ids = upstream15["row_ids"][upstream15["fit_mask"]]
frozen_train = upstream15["X"][upstream15["fit_mask"]].astype(np.float32)
EDA = upstream15["EDA"]
EDA_MANIFEST = upstream15["EDA_MANIFEST"]

hold_manifest = ROOT / ".cache/chunk-policy-review/program-chunk-qa-v1/MANIFEST.json"
reference_manifest = ROOT / ".cache/chunk-materialization/reference-v1/MANIFEST.json"

from fear_temperature.prechunk.chunking import MINILM_REVISION

hub = Path(os.environ.get("HF_HUB_CACHE", Path.home() / ".cache/huggingface/hub"))
snapshot = hub / "models--sentence-transformers--all-MiniLM-L6-v2/snapshots" / MINILM_REVISION
encoder_files = [
    "model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json",
    "vocab.txt", "special_tokens_map.json", "sentence_bert_config.json",
]
encoder_hashes = {name: sha256_file(snapshot / name) for name in encoder_files}
limit = int(json.loads((snapshot / "sentence_bert_config.json").read_text())["max_seq_length"])

SEEDS = (1729, 2211, 4021)
K_GRID = (8, 12, 16)
DENSITY_GRID = (20, 50, 100)
VALIDATION_TARGET = 5_000
SVD_RANK = 100
TOP_WORDS = 20
COHERENCE_WORDS = 10
BOOTSTRAP_DRAWS = 300
THREADS = 4
FAMILIES = {
    "svd_kmeans": K_GRID,
    "nmf": K_GRID,
    "lda": K_GRID,
    "minilm_kmeans": K_GRID,
    "bertopic": DENSITY_GRID,
    "svd_umap_hdbscan": DENSITY_GRID,
}
REFERENCE_CONFIG = "nmf-12"
packages = {
    name: version(name) for name in (
        "scikit-learn", "numpy", "scipy", "torch", "transformers", "bertopic",
        "hdbscan", "umap-learn", "joblib", "pyarrow", "pyspark",
    )
}
protocol = {
    "seeds": SEEDS,
    "grid": FAMILIES,
    "heldout_requested": VALIDATION_TARGET,
    "svd_rank": SVD_RANK,
    "top_words": TOP_WORDS,
    "coherence_words": COHERENCE_WORDS,
    "bootstrap_draws": BOOTSTRAP_DRAWS,
    "shared_evaluation_space": "l2_normalized_train_svd_seed_1729",
    "reference_configuration": REFERENCE_CONFIG,
    "minilm": {"revision": MINILM_REVISION, "total_token_limit": limit, "sha256": encoder_hashes},
    "fixed_threads": THREADS,
    "input_eda_manifest_sha256": sha256_file(EDA_MANIFEST),
    "hold_manifest_sha256": sha256_file(hold_manifest),
    "reference_checkpoint_sha256": sha256_file(reference_manifest),
    "code_sha256": digest([
        (cell.id, cell.source) for cell in nbformat.read(NOTEBOOK, as_version=4).cells
    ]),
    "packages": packages,
    "python": platform.python_version(),
    "platform": platform.platform(),
}
RUN_ROOT = ROOT / ".cache/document-model-comparison" / digest(protocol)[:20]
RUN_ROOT.mkdir(mode=0o700, parents=True, exist_ok=True)
input_dir = RUN_ROOT / "document_input"


def verify_snapshot(directory, expected_identity):
    data = json.loads((directory / "MANIFEST.json").read_text(encoding="utf-8"))
    # JSON round-tripping normalizes tuples in the plan.
    if digest(data["identity"]) != digest(expected_identity):
        raise ValueError(f"The saved snapshot belongs to different inputs: {directory.name}.")
    for entry in data["files"]:
        if sha256_file(directory / entry["path"]) != entry["sha256"]:
            raise ValueError(f"A saved artifact failed verification: {entry['path']}.")
    return data


def seal_snapshot(stage, target, run_identity, **measurements):
    manifest = {
        "identity": run_identity,
        **measurements,
        "files": [
            {"path": f.relative_to(stage).as_posix(), "sha256": sha256_file(f)}
            for f in sorted(stage.rglob("*")) if f.is_file() and not f.name.startswith(".")
        ],
    }
    (stage / "MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    if target.exists():
        raise FileExistsError(f"Another writer published {target.name}.")
    stage.rename(target)


print(f"Document-only comparison registry: {RUN_ROOT.relative_to(ROOT)}")
print(f"Planned fits: {len(SEEDS) * sum(len(grid) for grid in FAMILIES.values())}")


## Materialize the shared training and validation documents

Reuse the verified Notebook 05 complete document bodies, Notebook 13 lexical analyzer, vocabulary and IDF. Sample the validation panel by stable hash after hold exclusions and exact-body canonicalization. Verify training feature values and group separation. The prepared snapshot is private and reusable.

In [ ]:
if not input_dir.exists():
    from pyspark.ml.feature import CountVectorizerModel, IDFModel, Normalizer
    from pyspark.sql import Window
    from pyspark.sql import functions as F

    n12 = nbformat.read(ROOT / "notebooks/12_reference_chunk_materialization.ipynb", as_version=4)
    binder = next(c.source for c in n12.cells if c.id == "12-bind")
    reference = {"__name__": "__main__"}
    exec(compile(binder, "12_reference_chunk_materialization.ipynb:12-bind", "exec"), reference)
    spark = reference["spark"]
    try:
        linked = json.loads(EDA_MANIFEST.read_text(encoding="utf-8"))["identity"]
        if linked["document_handoff_manifest_sha256"] != reference["identity"]["source_handoff_manifest_sha256"]:
            raise ValueError("The document bodies do not match the original feature handoff.")
        parents = reference["parents"]
        train_keys = spark.createDataFrame(
            [
                (row, sha)
                for row, sha in zip(
                    train_ids.tolist(),
                    upstream15["cohort"]["clean_sha256"].to_numpy()[upstream15["fit_mask"]].tolist(),
                    strict=True,
                )
            ],
            "release_row long, clean_sha256 string",
        )
        fields = [
            "release_row", "clean_sha256", "clean_text", "ml_split", "source_host",
            "release_group_id", "format", "publication_year",
        ]
        train_docs = parents.where(F.col("ml_split") == "train").join(
            F.broadcast(train_keys), ["release_row", "clean_sha256"]
        ).select(*fields)
        excluded = reference["excluded_parents"].select("release_row").unionByName(
            reference["excluded_occurrences"].select("release_row")
        ).distinct()
        canonical = Window.partitionBy("clean_sha256").orderBy("release_row")
        val_docs = (
            parents.where(F.col("ml_split") == "validation")
            .where(~F.coalesce(F.col("cross_split_review_required"), F.lit(True)))
            .join(F.broadcast(excluded), "release_row", "left_anti")
            .withColumn("canonical_copy", F.row_number().over(canonical))
            .where(F.col("canonical_copy") == 1)
            .withColumn(
                "draw_key", F.sha2(F.concat_ws("|", F.lit("doc-validation-panel-v1"), "clean_sha256"), 256)
            )
            .orderBy("draw_key", "release_row").limit(VALIDATION_TARGET)
            .select(*fields)
        )
        if val_docs.join(F.broadcast(train_keys.select("clean_sha256")), "clean_sha256").limit(1).count():
            raise ValueError("Identical document text appears in both experimental partitions.")
        parents_selected = train_docs.unionByName(val_docs)
        split_count = {
            r.ml_split: r["count"] for r in parents_selected.groupBy("ml_split").count().collect()
        }
        if split_count != {"train": len(train_ids), "validation": VALIDATION_TARGET}:
            raise ValueError(f"The experimental document roster did not reconcile: {split_count}")

        # Apply the existing Notebook 13 analyzer to both document splits in one pass.
        n13 = nbformat.read(ROOT / "notebooks/13_document_eda_and_unsupervised_preparation.ipynb", as_version=4)
        lexical = next(c.source for c in n13.cells if c.id == "13-lexical")
        analyzer = lexical[lexical.index("segments = "):lexical.index("\nobserved =")]
        original = 'bodies.join(F.broadcast(selection), ["release_row", "clean_sha256"])'
        drop_text = '.drop("segments", "clean_text")'
        if analyzer.count(original) != 1 or analyzer.count(drop_text) != 1:
            raise ValueError("The bound lexical analyzer changed.")
        analyzer = analyzer.replace(original, "parents_selected").replace(drop_text, '.drop("segments")')
        token_scope = lexical[:lexical.index("\nbodies =")]
        from pyspark import StorageLevel

        lexical_scope = {"F": F, "parents_selected": parents_selected, "StorageLevel": StorageLevel}
        exec(compile(token_scope + "\n" + analyzer, "13_document_eda_and_unsupervised_preparation.ipynb:13-lexical", "exec"), lexical_scope)
        count_model = CountVectorizerModel.load(str(EDA / "count_vectorizer"))
        idf_model = IDFModel.load(str(EDA / "idf"))
        if count_model.vocabulary != vocab_words.tolist():
            raise ValueError("The saved Spark model uses a different vocabulary order.")
        features = Normalizer(inputCol="raw_tfidf", outputCol="tfidf").transform(
            idf_model.transform(count_model.transform(lexical_scope["tokenized"]))
        )
        with tempfile.TemporaryDirectory(prefix=".doc-input-", dir=RUN_ROOT) as tmp:
            stage = Path(tmp) / "snapshot"
            stage.mkdir(mode=0o700)
            (
                features.select(*fields, "term_counts", "tfidf")
                .write.mode("errorifexists").option("compression", "zstd")
                .partitionBy("ml_split").parquet(str(stage / "rows"))
            )
            seal_snapshot(stage, input_dir, protocol, split_count=split_count, source_policy="Notebook 13 analyzer")
    finally:
        spark.stop()

input_manifest = verify_snapshot(input_dir, protocol)


def load_split(split):
    paths = sorted((input_dir / "rows" / f"ml_split={split}").glob("*.parquet"))
    frame = pq.read_table(paths).sort_by("release_row")
    if len(frame) != input_manifest["split_count"][split]:
        raise ValueError("A staged split contains an unexpected number of documents.")

    def csr(column):
        vectors = frame[column].combine_chunks()
        if not np.all(vectors.field("type").to_numpy() == 0):
            raise ValueError("Expected sparse vectors.")
        ix = vectors.field("indices")
        val = vectors.field("values")
        if not np.array_equal(ix.offsets.to_numpy(), val.offsets.to_numpy()):
            raise ValueError("Vector index and value offsets differ.")
        matrix = csr_matrix(
            (val.flatten().to_numpy().astype(np.float32), ix.flatten().to_numpy(), ix.offsets.to_numpy()),
            shape=(len(frame), len(vocab_words)),
        )
        if not matrix.has_canonical_format:
            raise ValueError("Document feature vector indices are not canonical.")
        return matrix

    metadata = frame.select(
        ["release_row", "clean_sha256", "source_host", "release_group_id", "format", "publication_year"]
    ).to_pandas()
    if not metadata["release_row"].is_unique or metadata["release_group_id"].isna().any():
        raise ValueError("A prepared split has duplicate documents or lacks release groups.")
    return metadata, frame["clean_text"].to_pylist(), csr("term_counts"), csr("tfidf")


train_meta, train_text, train_counts, train_tfidf = load_split("train")
eval_meta, eval_text, eval_counts, eval_tfidf = load_split("validation")
if not np.array_equal(train_meta.release_row.to_numpy(), train_ids):
    raise ValueError("The staged training documents differ from the frozen Notebook 13 cohort.")
train_difference = train_tfidf - frozen_train
if train_difference.nnz and np.max(np.abs(train_difference.data)) > 1e-7:
    raise ValueError("Replayed training TF–IDF differs from the frozen Notebook 13 matrix.")
if set(train_meta.release_group_id) & set(eval_meta.release_group_id):
    raise ValueError("The comparison panel shares a release group with experimental training documents.")
display(pd.DataFrame(
    [(len(train_meta), train_counts.nnz), (len(eval_meta), eval_counts.nnz)],
    index=["Training", "Validation"], columns=["Documents", "Nonzero count features"],
))


## Prepare the shared document representations

Encode each parent with the pinned local MiniLM snapshot, accounting for all wordpieces. A one-off CPU encoding stage can dominate the initial runtime. Save the resulting vectors and coverage receipts. Use a seeded training-only SVD projection as the common validation geometry for internal cluster diagnostics.

In [ ]:
import joblib
import torch
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits
from transformers import AutoModel, AutoTokenizer

embedding_dir = RUN_ROOT / "minilm_document_vectors"
embedding_identity = {
    "protocol_sha256": digest(protocol),
    "cohort_manifest_sha256": sha256_file(input_dir / "MANIFEST.json"),
    "pooling": "all_wordpieces_mean_then_l2",
    "device": "cpu",
}


def encode_documents(texts, tokenizer, encoder):
    payload_limit = limit - tokenizer.num_special_tokens_to_add(pair=False)
    if tokenizer.padding_side != "right" or payload_limit <= 0 or tokenizer.build_inputs_with_special_tokens([]) != [
        tokenizer.cls_token_id, tokenizer.sep_token_id
    ]:
        raise ValueError("The pinned model did not expose the expected BERT token layout.")
    sums = np.zeros((len(texts), encoder.config.hidden_size), dtype=np.float64)
    token_counts = np.zeros(len(texts), dtype=np.int64)
    window_counts = np.zeros(len(texts), dtype=np.int32)
    for base in range(0, len(texts), 128):
        source_ids = tokenizer(
            texts[base:base + 128], add_special_tokens=False, truncation=False, verbose=False
        )["input_ids"]
        pending = []
        for idx, ids in enumerate(source_ids, base):
            token_counts[idx] = len(ids)
            for start in range(0, len(ids), payload_limit):
                pending.append((idx, ids[start:start + payload_limit]))
                window_counts[idx] += 1
        for start in range(0, len(pending), 48):
            batch = pending[start:start + 48]
            ids = [tokenizer.build_inputs_with_special_tokens(seq) for _, seq in batch]
            payload_length = np.array([len(seq) for _, seq in batch])
            model_inputs = tokenizer.pad(
                {"input_ids": ids}, padding=True, return_attention_mask=True, return_tensors="pt"
            )
            if model_inputs["input_ids"].shape[1] > limit:
                raise ValueError("A frozen encoder input exceeded its real token limit.")
            content = torch.arange(model_inputs["input_ids"].shape[1])[None, :]
            content = (content > 0) & (content <= torch.from_numpy(payload_length)[:, None])
            with torch.inference_mode():
                states = encoder(**model_inputs).last_hidden_state
                values = (states * content[:, :, None]).sum(1).numpy()
            for (idx, _), value in zip(batch, values, strict=True):
                sums[idx] += value
        if base % 2048 == 0:
            print(f"Encoded {min(base + 128, len(texts)):,} of {len(texts):,} documents.", flush=True)
    if not np.all(token_counts > 0):
        raise ValueError("The comparison roster includes a document with no model wordpieces.")
    if not np.all(window_counts == (token_counts + payload_limit - 1) // payload_limit):
        raise ValueError("The document encoding did not account for every model wordpiece.")
    pooled = normalize(sums / token_counts[:, None]).astype(np.float32)
    if not np.isfinite(pooled).all() or not np.all(np.sum(pooled ** 2, axis=1) > 0):
        raise ValueError("The frozen encoder produced a nonfinite or empty document vector.")
    return pooled, token_counts, window_counts


if not embedding_dir.exists():
    torch.set_num_threads(THREADS)
    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), local_files_only=True, trust_remote_code=False)
    encoder = AutoModel.from_pretrained(
        str(snapshot), local_files_only=True, trust_remote_code=False, use_safetensors=True
    ).eval()
    if limit > encoder.config.max_position_embeddings:
        raise ValueError("The recorded encoder limit exceeds its position capacity.")
    with tempfile.TemporaryDirectory(prefix=".doc-embedding-", dir=RUN_ROOT) as tmp:
        stage = Path(tmp) / "snapshot"
        stage.mkdir(mode=0o700)
        for split, text, meta in (("train", train_text, train_meta), ("validation", eval_text, eval_meta)):
            vectors, tokens, windows = encode_documents(text, tokenizer, encoder)
            np.save(stage / f"{split}.npy", vectors, allow_pickle=False)
            pd.DataFrame({
                "release_row": meta.release_row,
                "model_wordpieces": tokens,
                "internal_encoder_calls": windows,
            }).to_parquet(stage / f"{split}_coverage.parquet", index=False)
        seal_snapshot(stage, embedding_dir, embedding_identity)
    del encoder

verify_snapshot(embedding_dir, embedding_identity)
train_embedding = np.load(embedding_dir / "train.npy", allow_pickle=False)
eval_embedding = np.load(embedding_dir / "validation.npy", allow_pickle=False)
if len(train_embedding) != len(train_meta) or len(eval_embedding) != len(eval_meta):
    raise ValueError("The document-vector roster differs from the shared comparison inputs.")


@lru_cache(maxsize=len(SEEDS))
def svd_vectors(seed):
    stage_root = RUN_ROOT / f"svd-{seed}"
    identity = {"protocol_sha256": digest(protocol), "seed": seed, "input": "train_tfidf"}
    if not stage_root.exists():
        reducer = TruncatedSVD(n_components=SVD_RANK, n_iter=7, random_state=seed)
        with threadpool_limits(limits=THREADS):
            a = normalize(reducer.fit_transform(train_tfidf)).astype(np.float32)
            b = normalize(reducer.transform(eval_tfidf)).astype(np.float32)
        with tempfile.TemporaryDirectory(prefix=".svd-", dir=RUN_ROOT) as tmp:
            stage = Path(tmp) / "snapshot"
            stage.mkdir(mode=0o700)
            np.save(stage / "train.npy", a, allow_pickle=False)
            np.save(stage / "validation.npy", b, allow_pickle=False)
            joblib.dump(reducer, stage / "estimator.joblib", compress=3)
            seal_snapshot(stage, stage_root, identity)
    verify_snapshot(stage_root, identity)
    return (
        np.load(stage_root / "train.npy", allow_pickle=False),
        np.load(stage_root / "validation.npy", allow_pickle=False),
    )


with threadpool_limits(limits=THREADS):
    COMMON_GEOMETRY = svd_vectors(SEEDS[0])[1]
    reference_distances = pairwise_distances(COMMON_GEOMETRY, metric="euclidean").astype(np.float32)
np.fill_diagonal(reference_distances, 0)
print("Verified local document embeddings and the shared validation geometry.")


## Fit and preserve the model runs

Each completed run seals its fitted estimator, training and validation labels, native membership information, ranked terms, metrics, fit warnings and checksums. Shared encoder and SVD assets have their own receipts. Failures are logged by run ID. An interrupted Run All can be repeated from the relevant seed cell, which verifies and skips sealed runs and tries the remaining configurations.

The outputs are local research artifacts. Full estimator persistence includes the density reducers and clusterers. Open serialized estimators only from the local run snapshots in the recorded environment. [BERTopic serialization](https://maartengr.github.io/BERTopic/getting_started/serialization/serialization.html)

In [ ]:
import time
import traceback
import warnings
from itertools import combinations

from bertopic import BERTopic
from hdbscan import HDBSCAN, approximate_predict
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score, silhouette_samples,
)
from umap import UMAP

runs_dir = RUN_ROOT / "runs"
failures_dir = RUN_ROOT / "attempts"
for folder in (runs_dir, failures_dir):
    folder.mkdir(mode=0o700, exist_ok=True)

eval_binary = eval_counts.copy().tocsc()
eval_binary.data[:] = 1


def density_pipeline(param, seed):
    return (
        UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
             random_state=seed, transform_seed=seed, n_jobs=1),
        HDBSCAN(min_cluster_size=param, min_samples=10, metric="euclidean",
                prediction_data=True, core_dist_n_jobs=1),
    )


def rank_terms(weights):
    present = np.flatnonzero(weights > 0)
    order = np.lexsort((present, -weights[present]))[:TOP_WORDS]
    return [(vocab_words[i], float(weights[i])) for i in present[order]]


def centroid_descriptors(labels):
    return {
        int(label): rank_terms(np.asarray(train_tfidf[labels == label].mean(axis=0)).ravel())
        for label in np.unique(labels) if label >= 0
    }


def topic_contributions(raw, factors=None):
    weighted = raw if factors is None else raw * factors.sum(axis=1)[None, :]
    row_total = weighted.sum(axis=1, keepdims=True)
    shares = np.divide(weighted, row_total, out=np.zeros_like(weighted), where=row_total > 0)
    labels = np.where(row_total.ravel() > 0, shares.argmax(axis=1), -1)
    return labels, shares


def fit_model(family, param, seed):
    auxiliary_metrics = {}
    descriptors = None
    if family in {"svd_kmeans", "minilm_kmeans"}:
        left, right = svd_vectors(seed) if family == "svd_kmeans" else (train_embedding, eval_embedding)
        model = KMeans(n_clusters=param, n_init=10, random_state=seed, algorithm="lloyd")
        train_label = model.fit_predict(left)
        eval_label = model.predict(right)
        memberships = {"train_cluster_distances": model.transform(left), "validation_cluster_distances": model.transform(right)}
        trained = {"estimator": model}
    elif family == "nmf":
        model = NMF(n_components=param, init="nndsvdar", solver="cd", max_iter=250,
                    tol=1e-4, random_state=seed, shuffle=False)
        a = model.fit_transform(train_tfidf)
        b = model.transform(eval_tfidf)
        train_label, tr = topic_contributions(a, model.components_)
        eval_label, ev = topic_contributions(b, model.components_)
        descriptors = {i: rank_terms(row) for i, row in enumerate(model.components_)}
        memberships = {"train_raw_weights": a, "validation_raw_weights": b,
                       "train_relative_contributions": tr, "validation_relative_contributions": ev}
        auxiliary_metrics["train_reconstruction_error"] = float(model.reconstruction_err_)
        auxiliary_metrics["fit_iterations"] = int(model.n_iter_)
        trained = {"estimator": model}
    elif family == "lda":
        model = LatentDirichletAllocation(n_components=param, max_iter=20, learning_method="batch",
                                         random_state=seed, n_jobs=1, evaluate_every=-1)
        a = model.fit_transform(train_counts)
        b = model.transform(eval_counts)
        train_label, _ = topic_contributions(a)
        eval_label, _ = topic_contributions(b)
        train_label[np.diff(train_counts.indptr) == 0] = -1
        eval_label[np.diff(eval_counts.indptr) == 0] = -1
        descriptors = {i: rank_terms(row) for i, row in enumerate(model.components_)}
        memberships = {"train_topic_probabilities": a, "validation_topic_probabilities": b}
        auxiliary_metrics["validation_perplexity"] = float(model.perplexity(eval_counts))
        trained = {"estimator": model}
    elif family == "svd_umap_hdbscan":
        left, right = svd_vectors(seed)
        reducer, model = density_pipeline(param, seed)
        train_label = model.fit_predict(reducer.fit_transform(left))
        if np.any(train_label >= 0):
            eval_label, strength = approximate_predict(model, reducer.transform(right))
        else:
            eval_label, strength = np.full(len(eval_meta), -1), np.zeros(len(eval_meta))
        memberships = {"train_membership_strength": model.probabilities_,
                       "validation_membership_strength": strength}
        trained = {"estimator": model, "umap": reducer}
    elif family == "bertopic":
        reducer, clusterer = density_pipeline(param, seed)
        model = BERTopic(
            embedding_model=None, umap_model=reducer, hdbscan_model=clusterer,
            vectorizer_model=CountVectorizer(
                vocabulary=vocab_index, ngram_range=(1, 2),
                token_pattern=r"(?u)[^\W_]+(?:['’][^\W_]+)*",
            ),
            top_n_words=TOP_WORDS, calculate_probabilities=False, verbose=False,
        )
        train_label, a = model.fit_transform(train_text, embeddings=train_embedding)
        train_label = np.asarray(train_label)
        if np.any(train_label >= 0):
            eval_label, b = model.transform(eval_text, embeddings=eval_embedding)
        else:
            eval_label, b = np.full(len(eval_meta), -1), np.zeros(len(eval_meta))
        descriptors = {
            int(lab): [(word, float(score)) for word, score in words if word in vocab_index and score > 0]
            for lab, words in model.get_topics().items() if lab >= 0
        }
        memberships = {
            name: values for name, values in
            (("train_membership_strength", a), ("validation_membership_strength", b)) if values is not None
        }
        trained = {"estimator": model}
    else:
        raise ValueError(f"Unexpected registered family: {family}")
    train_label = np.asarray(train_label, dtype=np.int32)
    eval_label = np.asarray(eval_label, dtype=np.int32)
    if train_label.shape != (len(train_meta),) or eval_label.shape != (len(eval_meta),):
        raise ValueError("The model did not return one assignment per parent.")
    if descriptors is None:
        descriptors = centroid_descriptors(train_label)
    return trained, train_label, eval_label, memberships, descriptors, auxiliary_metrics


def validation_npmi(descriptors):
    values = []
    n = len(eval_meta)
    for terms in descriptors.values():
        words = [vocab_index[word] for word, _ in terms[:COHERENCE_WORDS]]
        if len(words) < 2:
            continue
        block = eval_binary[:, words]
        pair_counts = (block.T @ block).toarray()
        marginal = np.diag(pair_counts)
        for i, j in combinations(range(len(words)), 2):
            joint = pair_counts[i, j]
            if joint == 0:
                values.append(-1.0)
            elif joint == n:
                values.append(0.0)
            else:
                values.append(float(
                    np.log((float(joint) * n) / (float(marginal[i]) * marginal[j]))
                    / -np.log(joint / n)
                ))
    return float(np.mean(values)) if values else None


def assess(labels, terms):
    valid = labels >= 0
    used = labels[valid]
    cardinality = len(np.unique(used))
    silhouettes = np.full(len(labels), np.nan, dtype=np.float32)
    ch, db, silhouette = None, None, None
    if 1 < cardinality < len(used):
        subset = reference_distances if valid.all() else reference_distances[np.ix_(valid, valid)]
        silhouettes[valid] = silhouette_samples(subset, used, metric="precomputed")
        ch = float(calinski_harabasz_score(COMMON_GEOMETRY[valid], used))
        db = float(davies_bouldin_score(COMMON_GEOMETRY[valid], used))
        silhouette = float(np.mean(silhouettes[valid]))
    words = [word for values in terms.values() for word, _ in values[:TOP_WORDS]]
    return {
        "validation_topics": cardinality,
        "validation_coverage": float(valid.mean()),
        "validation_noise_rate": float((~valid).mean()),
        "validation_silhouette_common_svd": silhouette,
        "validation_calinski_harabasz": ch,
        "validation_davies_bouldin": db,
        "validation_npmi": validation_npmi(terms),
        "topic_diversity": len(set(words)) / len(words) if words else None,
    }, silhouettes


def fit_and_save(seed, family, param):
    run_id = f"{family}-{param}-seed{seed}"
    destination = runs_dir / run_id
    run_identity = {"protocol_sha256": digest(protocol), "seed": seed, "family": family, "parameter": param}
    if destination.exists():
        verify_snapshot(destination, run_identity)
        return "verified"
    started = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix=".fit-", dir=runs_dir) as tmp:
        stage = Path(tmp) / "snapshot"
        stage.mkdir(mode=0o700)
        with threadpool_limits(limits=THREADS), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model, train_label, eval_label, membership, terms, family_metrics = fit_model(family, param, seed)
            elapsed = time.perf_counter() - started
            metrics, silhouettes = assess(eval_label, terms)
        metrics.update(family_metrics)
        metrics.update({
            "fit_seconds": elapsed, "train_topics": int(np.unique(train_label[train_label >= 0]).size),
            "train_documents": len(train_meta), "validation_documents": len(eval_meta),
        })
        joblib.dump(model, stage / "estimator.joblib", compress=3)
        np.savez_compressed(
            stage / "memberships.npz",
            train_release_rows=train_meta.release_row.to_numpy(),
            validation_release_rows=eval_meta.release_row.to_numpy(),
            **membership,
        )
        pd.DataFrame({"release_row": train_meta.release_row, "topic": train_label}).to_parquet(
            stage / "train_assignments.parquet", index=False
        )
        pd.DataFrame({
            "release_row": eval_meta.release_row, "topic": eval_label, "common_svd_silhouette": silhouettes,
        }).to_parquet(stage / "validation_assignments.parquet", index=False)
        pd.DataFrame(
            [(topic, rank, word, value) for topic, entries in terms.items()
             for rank, (word, value) in enumerate(entries, 1)],
            columns=["topic", "rank", "term", "weight"],
        ).to_csv(stage / "topic_terms.csv", index=False)
        pd.DataFrame(
            [
                (int(label), int(count)) for label, count in zip(
                    *np.unique(eval_label, return_counts=True), strict=True
                )
            ], columns=["topic", "validation_documents"]
        ).to_csv(stage / "validation_topic_sizes.csv", index=False)
        seal_snapshot(
            stage, destination, run_identity, metrics=metrics,
            warnings=sorted({str(item.message) for item in caught}),
        )
    return "completed"


def run_seed(seed):
    if seed not in SEEDS:
        raise ValueError("This seed is not in the declared registry.")
    for family, grid in FAMILIES.items():
        for param in grid:
            run_id = f"{family}-{param}-seed{seed}"
            tick = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
            print(f"{tick}  {run_id}", flush=True)
            try:
                print(f"  {fit_and_save(seed, family, param)}", flush=True)
            except Exception as exc:
                from uuid import uuid4

                fail = failures_dir / f"{run_id}-{uuid4().hex[:10]}.json"
                with fail.open("x", encoding="utf-8") as handle:
                    json.dump(
                        {"run_id": run_id, "started_at_utc": tick, "error": repr(exc),
                         "traceback": traceback.format_exc()},
                        handle, indent=2,
                    )
                print(f"  failed. See {fail.relative_to(ROOT)}", flush=True)


### Seed 1729 · all 18 configurations

In [ ]:
run_seed(1729)

### Seed 2211 · all 18 configurations

In [ ]:
run_seed(2211)

### Seed 4021 · all 18 configurations

In [ ]:
run_seed(4021)

## Compare configurations on the same held-out documents

Report seed means and standard deviations, common-assignment ARI, matched top-term overlap and validation coverage. Use a paired grouped bootstrap over validation documents for ARI intervals and comparisons to the predeclared NMF-12 silhouette reference. The bootstrap varies evaluation groups with the fitted models held fixed. The paired silhouette contrast conditions on documents assigned in every seed for both configurations.

Reconstruction error and perplexity remain in the per-run ledger. The comparison does not assign a production winner. Use the common-axis results with representative evidence, source concentration checks and any later training-resample sensitivity study.

In [ ]:
from scipy.optimize import linear_sum_assignment
from statsmodels.stats.multitest import multipletests

register = []
per_run = []
validated_runs = {}
eval_ids = eval_meta.release_row.to_numpy()
for family, grid in FAMILIES.items():
    for param in grid:
        config = f"{family}-{param}"
        for seed in SEEDS:
            run_id = f"{config}-seed{seed}"
            folder = runs_dir / run_id
            failures = sorted(failures_dir.glob(f"{run_id}-*.json"))
            if not folder.exists():
                register.append({
                    "configuration": config, "family": family, "parameter": param,
                    "seed": seed, "status": "failed" if failures else "pending",
                    "failure_attempts": len(failures),
                })
                continue
            saved = verify_snapshot(folder, {
                "protocol_sha256": digest(protocol), "seed": seed, "family": family, "parameter": param,
            })
            status = {"configuration": config, "family": family, "parameter": param, "seed": seed,
                      "status": "complete", "failure_attempts": len(failures)}
            register.append(status)
            per_run.append({**status, "warnings": len(saved["warnings"]), **saved["metrics"]})
            assignments = pd.read_parquet(folder / "validation_assignments.parquet")
            if not np.array_equal(assignments.release_row.to_numpy(), eval_ids):
                raise ValueError("The comparison assignment roster changed.")
            words = pd.read_csv(folder / "topic_terms.csv", keep_default_na=False)
            validated_runs[config, seed] = {
                "labels": assignments.topic.to_numpy(),
                "silhouette": assignments.common_svd_silhouette.to_numpy(),
                "descriptors": {
                    label: set(rows.sort_values("rank").term.head(TOP_WORDS))
                    for label, rows in words.groupby("topic")
                },
                "manifest_sha256": sha256_file(folder / "MANIFEST.json"),
            }

registry_table = pd.DataFrame(register)
metric_columns = [
    "fit_seconds", "validation_topics", "validation_coverage", "validation_noise_rate",
    "validation_silhouette_common_svd", "validation_calinski_harabasz", "validation_davies_bouldin",
    "validation_npmi", "topic_diversity",
]
run_table = pd.DataFrame(per_run).reindex(columns=[
    *registry_table.columns, "warnings", *metric_columns,
    "train_topics", "train_documents", "validation_documents", "train_reconstruction_error",
    "fit_iterations", "validation_perplexity",
])
overview = run_table.groupby(["configuration", "family", "parameter"], sort=True).agg(
    seed_runs=("seed", "size"),
    **{
        f"{metric}_{stat}": (metric, stat)
        for metric in metric_columns for stat in ("mean", "std")
    },
).reset_index()

group_ids = pd.factorize(eval_meta.release_group_id, sort=True)[0]
group_order = np.argsort(group_ids, kind="stable")
group_offsets = np.flatnonzero(np.diff(group_ids[group_order])) + 1
group_rows = np.split(group_order, group_offsets)
bootstrap_rng = np.random.default_rng(91721)
draws = [
    np.concatenate([group_rows[i] for i in bootstrap_rng.integers(0, len(group_rows), len(group_rows))])
    for _ in range(BOOTSTRAP_DRAWS)
]


def aligned_word_overlap(left, right):
    if not left or not right:
        return np.nan
    matrix = np.array([
        [len(a & b) / len(a | b) if a or b else 0.0 for b in right.values()]
        for a in left.values()
    ])
    ai, bi = linear_sum_assignment(-matrix)
    return matrix[ai, bi].sum() / max(len(left), len(right))


def comparable_ari(left, right):
    common = (left >= 0) & (right >= 0)
    if len(np.unique(left[common])) < 2 or len(np.unique(right[common])) < 2:
        return np.nan
    return adjusted_rand_score(left[common], right[common])


pair_records = []
for config in sorted(registry_table.configuration.unique()):
    for a, b in combinations(SEEDS, 2):
        first, second = validated_runs.get((config, a)), validated_runs.get((config, b))
        if first is None or second is None:
            continue
        left, right = first["labels"], second["labels"]
        # These intervals vary comparison documents with fitted models held fixed.
        boot = np.asarray([comparable_ari(left[index], right[index]) for index in draws])
        boot = boot[np.isfinite(boot)]
        bounds = np.quantile(boot, (0.025, 0.975)) if len(boot) else (np.nan, np.nan)
        pair_records.append({
            "configuration": config,
            "seed_pair": f"{a}/{b}",
            "common_assignment_coverage": float(np.mean((left >= 0) & (right >= 0))),
            "ari_common_assigned": comparable_ari(left, right),
            "ari_bootstrap_low": bounds[0],
            "ari_bootstrap_high": bounds[1],
            "aligned_top20_jaccard": aligned_word_overlap(first["descriptors"], second["descriptors"]),
        })
stability = pd.DataFrame(pair_records, columns=[
    "configuration", "seed_pair", "common_assignment_coverage", "ari_common_assigned",
    "ari_bootstrap_low", "ari_bootstrap_high", "aligned_top20_jaccard",
])
overall_stability = stability.groupby("configuration").agg(
    seed_pairs=("seed_pair", "size"),
    ari_mean=("ari_common_assigned", "mean"),
    ari_sd=("ari_common_assigned", "std"),
    term_overlap_mean=("aligned_top20_jaccard", "mean"),
    pair_coverage_mean=("common_assignment_coverage", "mean"),
).reset_index()
scorecard = (
    registry_table.groupby(["configuration", "family", "parameter"]).size().rename("planned_seeds")
    .reset_index().merge(overview, on=["configuration", "family", "parameter"], how="left")
    .merge(overall_stability, on="configuration", how="left")
)
scorecard["seed_runs"] = scorecard.seed_runs.fillna(0).astype(int)
scorecard["seed_pairs"] = scorecard.seed_pairs.fillna(0).astype(int)


def mean_silhouettes(config):
    values = [validated_runs.get((config, seed)) for seed in SEEDS]
    if any(run is None for run in values):
        return None
    # A document contributes only if it was assigned in every registered seed.
    raw = np.vstack([item["silhouette"] for item in values])
    return np.mean(raw, axis=0)


def paired_silhouette(left, right, seed):
    common = np.isfinite(left) & np.isfinite(right)
    if not np.any(common):
        return None
    selected_groups, _ = pd.factorize(group_ids[common], sort=True)
    delta = (left[common] - right[common]).astype(np.float64)
    counts = np.bincount(selected_groups)
    sums = np.bincount(selected_groups, weights=delta)
    if len(counts) < 2:
        return None
    total = len(delta)
    observed = delta.mean()
    rng = np.random.default_rng(seed)
    selections = rng.integers(0, len(counts), (BOOTSTRAP_DRAWS, len(counts)))
    boot = sums[selections].sum(axis=1) / counts[selections].sum(axis=1)
    low, high = np.quantile(boot, (0.025, 0.975))
    signs = 2 * rng.integers(0, 2, (999, len(counts)), dtype=np.int8) - 1
    flipped = (signs @ sums) / total
    pval = (1 + np.count_nonzero(np.abs(flipped) >= abs(observed))) / (1 + len(flipped))
    return {
        "paired_documents": total, "paired_release_groups": len(counts),
        "silhouette_mean_difference": observed,
        "bootstrap_95_low": low, "bootstrap_95_high": high,
        "paired_sign_flip_p": pval,
    }


contrasts = []
baseline = mean_silhouettes(REFERENCE_CONFIG)
if baseline is not None:
    for i, config in enumerate(sorted(registry_table.configuration.unique())):
        if config == REFERENCE_CONFIG:
            continue
        candidate = mean_silhouettes(config)
        evidence = None if candidate is None else paired_silhouette(candidate, baseline, 17007 + i)
        if evidence is not None:
            contrasts.append({"configuration": config, "reference": REFERENCE_CONFIG, **evidence})
contrast_table = pd.DataFrame(contrasts, columns=[
    "configuration", "reference", "paired_documents", "paired_release_groups",
    "silhouette_mean_difference", "bootstrap_95_low", "bootstrap_95_high", "paired_sign_flip_p",
])
contrast_table["holm_adjusted_p"] = (
    multipletests(contrast_table.paired_sign_flip_p, method="holm")[1]
    if not contrast_table.empty else pd.Series(dtype=float)
)
completed_count = len(validated_runs)
total_count = len(registry_table)
print(f"Verified completed fits: {completed_count} / {total_count}")
display(scorecard[["configuration", "seed_runs", "validation_coverage_mean",
                   "validation_npmi_mean", "ari_mean", "fit_seconds_mean"]])


## Export the comparison charts

Save the scorecard, seed stability heatmaps and paired contrast plot as PNG and PDF, together with the source CSVs. The registry distinguishes verified results from incomplete or failed candidates.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image

palette = dict(zip(FAMILIES, ["#235789", "#b65d29", "#477847", "#775687", "#007a80", "#a34762"], strict=True))
ordered = scorecard.sort_values(["family", "parameter"]).reset_index(drop=True)
figures = {}
fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)
for ax, metric, label in zip(
    axes.flat,
    ["validation_npmi", "validation_silhouette_common_svd", "validation_coverage", "fit_seconds"],
    ["Held-out NPMI", "Silhouette in shared SVD space", "Validation coverage", "Fit wall seconds"],
    strict=True,
):
    ranked = ordered[ordered[f"{metric}_mean"].notna()]
    for r in ranked.itertuples(index=True):
        mean = getattr(r, f"{metric}_mean")
        sd = getattr(r, f"{metric}_std")
        ax.errorbar(mean, r.Index, xerr=sd if np.isfinite(sd) else 0.0, fmt="o",
                    color=palette[r.family], capsize=3)
    ax.set_yticks(ordered.index, ordered.configuration)
    ax.invert_yaxis()
    ax.set_xlabel(label)
    ax.grid(axis="x", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Document model comparison · {completed_count}/{total_count} verified fits · seed means and SD")
figures["01_scorecard"] = fig

if not stability.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 9), constrained_layout=True)
    for ax, metric, caption, vmin in (
        (axes[0], "ari_common_assigned", "Document ARI on common assignments", -1),
        (axes[1], "aligned_top20_jaccard", "Aligned top-20 term Jaccard", 0),
    ):
        table = stability.pivot(index="configuration", columns="seed_pair", values=metric)
        visual = ax.imshow(table.to_numpy(), aspect="auto", cmap="viridis", vmin=vmin, vmax=1)
        ax.set_yticks(range(len(table)), table.index)
        ax.set_xticks(range(len(table.columns)), table.columns, rotation=20)
        ax.set_title(caption)
        fig.colorbar(visual, ax=ax, shrink=0.7)
    figures["02_seed_stability"] = fig

if not contrast_table.empty:
    d = contrast_table.sort_values("silhouette_mean_difference").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
    ax.axvline(0, linewidth=1, color="#68737a")
    ax.hlines(d.index, d.bootstrap_95_low, d.bootstrap_95_high, color="#225a83")
    ax.scatter(d.silhouette_mean_difference, d.index, color="#225a83")
    ax.set_yticks(d.index, d.configuration)
    ax.set_xlabel(f"Paired silhouette difference from {REFERENCE_CONFIG} · 95% grouped bootstrap CI")
    ax.set_title("Shared validation documents assigned across all three seeds")
    ax.spines[["top", "right"]].set_visible(False)
    figures["03_paired_silhouette"] = fig

report_identity = {
    "protocol_sha256": digest(protocol),
    "successful_runs": {
        f"{config}-seed{seed}": row["manifest_sha256"] for (config, seed), row in validated_runs.items()
    },
    "recorded_attempt_files": {
        file.name: sha256_file(file) for file in sorted(failures_dir.glob("*.json"))
    },
}
report_dir = RUN_ROOT / f"report-{digest(report_identity)[:14]}"
if not report_dir.exists():
    with tempfile.TemporaryDirectory(prefix=".comparison-", dir=RUN_ROOT) as tmp:
        stage = Path(tmp) / "snapshot"
        stage.mkdir(mode=0o700)
        for name, frame in (
            ("run_registry", registry_table), ("run_metrics", run_table), ("scorecard", scorecard),
            ("seed_stability", stability), ("paired_silhouette", contrast_table),
        ):
            frame.to_csv(stage / f"{name}.csv", index=False)
        for name, fig in figures.items():
            fig.savefig(stage / f"{name}.png", dpi=180, facecolor="white")
            fig.savefig(stage / f"{name}.pdf", facecolor="white")
        seal_snapshot(stage, report_dir, report_identity, completed=completed_count, planned=total_count)
verify_snapshot(report_dir, report_identity)
for name, fig in figures.items():
    plt.close(fig)
    display(Image(filename=str(report_dir / f"{name}.png"), width=900))
print(f"Comparison output: {report_dir.relative_to(ROOT)}")
if completed_count != total_count:
    print("The report is partial. Rerun the required seed cells to retry missing fits.")
